In [1]:
import os, secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

### QKD simulation (BB84 Simulation)

In [2]:
#defining a function with 127 bits to be sent to Bob by Alice
def bb84_simulate(n_bits=127):
    abits = [secrets.randbits(1) for _ in range(n_bits)]        #cryptographically secure random bits.
    abases = [secrets.randbits(1) for _ in range(n_bits)]
    bbases = [secrets.randbits(1) for _ in range(n_bits)]  
    #Bob doesn’t know Alice’s bases, so he randomly chooses his own measurement bases. half the time, he’ll “guess wrong.”
    bgoodbits = [abits[i] if abases[i]==bbases[i] else secrets.randbelow(2) for i in range(n_bits)]
    #If Bob’s basis matches Alice’s (alice_bases[i] == bob_bases[i]), Bob measures the correct bit (alice_bits[i])
    #When Bob measures in the wrong basis, quantum mechanics ensures his result is random and destroys the original state.
    sifted = [i for i in range(n_bits) if abases[i]==bbases[i]]
    #After transmission, Alice and Bob publicly compare their bases (not the bits!).
    #They keep only the indices where the bases matched — this process is called sifting.
    #Result: list of positions where both used the same basis.
    key_bits = [abits[i] for i in sifted]
    #Alice takes the bits corresponding to those matching bases — this becomes her sifted key.
    #Bob can do the same using his bob_results to get his version of the key.
    return key_bits

In [3]:
#This function converts a list of bits into actual byte values (which are needed for cryptographic keys).
#AES and HKDF work on bytes, not individual bits.
def bits_to_bytes(bits):
    b = bytearray()
    for i in range(0, len(bits), 8):     #Looping through the bits in groups of 8
        byte = 0
        for j in range(8):                #Looping 8 times (for the 8 bits that make up a single byte).
            if i+j < len(bits):
                byte = (byte << 1) | bits[i+j]      #byte << 1 → shifts bits to the left by one position; bits[i+j] → adds the new bit (0 or 1) into the least significant position.
            else:
                byte = (byte << 1)
        b.append(byte)
        #Adding the newly formed byte to the bytearray.
    return bytes(b)
    #Converts the mutable bytearray into an immutable bytes object (like b'\xaa\x7f\x1d...') — this is what cryptographic functions expect.

In [4]:
#this is the key derivation step, where you take the raw random bits from your QKD simulation
#and turn them into a cryptographically strong encryption key that you can safely use with AES.
def derive_key(bits):
    ikm = bits_to_bytes(bits)      #Input Keying Material — to describe the initial random data used to derive a key.
    hkdf = HKDF(algorithm=hashes.SHA256(), length=32, salt=None, info=b"qkd-key", backend=default_backend())
    #HMAC-based Key Derivation Function
    return hkdf.derive(ikm)
#If your QKD gave 256 bits like [1,0,1,1,0,...], this converts them into 32 bytes (since 8 bits = 1 byte).

In [5]:
#goal: convert these raw, somewhat noisy random bits recieved from QKD sifting into a clean, uniform cryptographic key.

### AES-GCM (Used to encryot the PT)

In [6]:
plaintext = b"This is a scret message that I live in IIT Ropar"    #data you want to protect
#Represents the actual AI model data or parameters in your larger project.
#written as a byte string (prefix b"") since AES operates on bytes
print("Plaintext:", plaintext.decode())                       #.decode() simply prints it as a human-readable string.



Plaintext: This is a scret message that I live in IIT Ropar


In [7]:
# Generate shared key from simulated QKD
key_bits = bb84_simulate(256)                 #256 bits act as the quantum-generated key material; these bits come from photons exchanged through a quantum channel.
session_key = derive_key(key_bits)     #Session keys are temporary keys valid for a particular session, once used, discarded
print("Derived key:", session_key.hex())
#This key is shared secretly between sender and receiver — no one else knows it.

Derived key: d843fa11a95d9956571e5dd2af5df21c49d0e2c194412d971c798e3bffc5a4e6


In [8]:
aesgcm = AESGCM(session_key)            #initializes AES in Galois/Counter Mode (GCM) using your derived session key.
nonce = os.urandom(12)                #generates a random 12-byte nonce (IV). It ensures that even if the same key is reused, the ciphertext will be different.
ciphertext = aesgcm.encrypt(nonce, plaintext, None)
print("Ciphertext:", ciphertext.hex())

Ciphertext: e3bf15ee49761f45a44e90ec47fee24c8e3ee7ec8c27139c8e557b7d6cef58f3f1c376889cbad833dccb23c61ab61ff5a7dfbdb233c146bc979bbc771e26b1df


In [9]:
#decrypt using same key
decrypted = AESGCM(session_key).decrypt(nonce, ciphertext, None)
print("Decrypted:", decrypted.decode())

Decrypted: This is a scret message that I live in IIT Ropar
